# Pipeline ETL: Enriquecimento de Dados Bancários com IA Generativa
Este notebook demonstra um fluxo completo de **ETL (Extração, Transformação e Carregamento)**.
O objetivo principal é ler uma base de clientes simulada, utilizar a Inteligência Artificial (OpenAI) para criar mensagens de marketing ultra personalizadas sobre investimentos e salvar esses dados enriquecidos para uso futuro.

**Tecnologias aplicadas:** Python, Pandas (Manipulação de Dados) e API da OpenAI (GPT).

In [ ]:
# Instalação da biblioteca da OpenAI (caso ainda não esteja instalada no ambiente)
!pip install openai==0.28

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 2.6 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.32.0
    Uninstalling openai-2.32.0:
      Successfully uninstalled openai-2.32.0



##1. Extração (Extract): Lendo e Preparando a Base de Clientes
Nesta primeira etapa, vamos extrair os dados bancários do nosso arquivo local (`SDW2023.csv`) utilizando o Pandas. Em seguida, vamos moldar essa tabela para uma estrutura de dicionários, preparando o "terreno" para receber as mensagens geradas pela inteligência artificial.

In [ ]:
import pandas as pd
import json

# Lê o arquivo CSV local (lembre-se de fazer o upload do arquivo SDW2023.csv para o Colab)
df = pd.read_csv('SDW2023.csv')

# Converte o DataFrame para uma lista de dicionários
users_raw = df.to_dict(orient='records')

# Molda os dados para a estrutura esperada
users = []
for row in users_raw:
    user = {
        "id": row['UserID'],
        "name": row['Name'],
        "account": {
            "number": row['AccountNumber'],
            "agency": str(row['Agency']).zfill(4),
            "balance": row['Balance'],
            "limit": row['Limit']
        },
        "card": {
            "number": row['CardNumber'],
            "limit": row['CardLimit']
        },
        "features": [],
        "news": [] # Preparamos a lista vazia que receberá a mensagem da IA
    }
    users.append(user)

print("Extração concluída com sucesso! Visualizando os dois primeiros usuários:")
print(json.dumps(users[:2], indent=2, ensure_ascii=False))

Extração concluída com sucesso! Visualizando os dois primeiros usuários:
[
  {
    "id": 1,
    "name": "Pyterson",
    "account": {
      "number": "00001-1",
      "agency": "0001",
      "balance": 0.0,
      "limit": 500.0
    },
    "card": {
      "number": "**** **** **** 1111",
      "limit": 1000.0
    },
    "features": [],
    "news": []
  },
  {
    "id": 2,
    "name": "Pip",
    "account": {
      "number": "00002-2",
      "agency": "0001",
      "balance": 150.5,
      "limit": 500.0
    },
    "card": {
      "number": "**** **** **** 2222",
      "limit": 1000.0
    },
    "features": [],
    "news": []
  }
]


##2. Transformação (Transform): Gerando Mensagens com OpenAI
Aqui acontece a mágica. Vamos iterar sobre a nossa lista de clientes e enviar o nome de cada um para a API da OpenAI (GPT). A IA atuará como uma especialista em marketing, gerando uma mensagem curta e persuasiva sobre a importância de investir, que será anexada ao perfil de cada cliente.

*Nota: É necessário inserir uma API Key válida da OpenAI no código abaixo.*

In [ ]:
import openai

# Substitua pela sua API Key real
openai.api_key = 'SUA_API_KEY_AQUI'

def generate_ai_news(user):
    try:
        # Usando gpt-3.5-turbo que é mais rápido e barato para testes
        completion = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",
            messages=[
                {
                    "role": "system",
                    "content": "Você é um especialista em marketing bancário."
                },
                {
                    "role": "user",
                    "content": f"Crie uma mensagem para {user['name']} sobre a importância dos investimentos (máximo de 100 caracteres)"
                }
            ]
        )
        return completion.choices[0].message.content.strip('\"')
    except Exception as e:
        return f"Invista hoje mesmo para garantir o seu futuro, {user['name']}!" # Mensagem de fallback caso a API falhe

# Gerando as mensagens para cada usuário
for user in users:
    news = generate_ai_news(user)
    print(f"Mensagem gerada para {user['name']}: {news}")
    user['news'].append({
        "icon": "https://digitalinnovationone.github.io/santander-dev-week-2023-api/icons/credit.svg",
        "description": news
    })

Mensagem gerada para Pyterson: Invista hoje mesmo para garantir o seu futuro, Pyterson!
Mensagem gerada para Pip: Invista hoje mesmo para garantir o seu futuro, Pip!
Mensagem gerada para Pep: Invista hoje mesmo para garantir o seu futuro, Pep!
Mensagem gerada para Alana: Invista hoje mesmo para garantir o seu futuro, Alana!
Mensagem gerada para Caio: Invista hoje mesmo para garantir o seu futuro, Caio!


##3. Carregamento (Load): Consolidando e Exportando o Novo Dataset
Com os dados originais enriquecidos pelas mensagens da IA, precisamos salvar esse resultado. Como não estamos consumindo a API original do projeto para enviar esses dados de volta, vamos exportar nossa lista final para um arquivo `JSON`. Isso simula a gravação em um banco de dados NoSQL e conclui nosso ciclo ETL.

In [ ]:
# Salvando o resultado final localmente
arquivo_saida = 'SDW2023_Atualizado.json'

with open(arquivo_saida, 'w', encoding='utf-8') as f:
    json.dump(users, f, ensure_ascii=False, indent=2)

print(f"\nCarregamento concluído! O arquivo '{arquivo_saida}' foi salvo com sucesso na raiz do seu Colab.")


Carregamento concluído! O arquivo 'SDW2023_Atualizado.json' foi salvo com sucesso na raiz do seu Colab.
